# 🧩 Seg 균열 검증 (Kaggle 병렬)  ·  YOLOv8s-seg

**목적**: 데스크탑이 50% BBox 재학습 도는 동안, 캐글 P100로 **균열 세그멘테이션 서브셋**을 빠르게 학습·검증해
마스크 mAP·비주얼로 "seg가 박스보다 나은가"를 판단한다. (본 학습 아님, 실익 검증용)

### 사전 준비
1. 데스크탑에서 `seg_make_kaggle_subset.py` 실행 → `seg_subset.zip` 생성.
2. 캐글 우측 **Add Input → Upload → New Dataset** 으로 그 zip 업로드.
3. 우측 **Settings → Accelerator = GPU (P100)**, **Internet = On**.
4. 위에서부터 순서대로 Run.

### 읽는 법
- `mask mAP50` 이 핵심 지표. 박스 크랙 모델(0.179)과 **직접 비교 금지**(박스 vs 마스크 = 다른 지표).
- 판단은 숫자 + 아래 **예측 비주얼**(마스크가 균열선을 얼마나 정확히 덮는가)로.


In [ ]:
# [1] 파라미터 (검증용 소량 세팅)
EPOCHS   = 60          # 서브셋이라 60ep면 수렴 경향 확인 충분 (P100 ≈ 1~2h)
IMGSZ    = 640
BATCH    = 16          # P100 16GB 여유. OOM 시 8.
MODEL    = "yolov8s-seg.pt"
PROJECT  = "/kaggle/working/runs"
NAME     = "seg_crack_val"


In [ ]:
# [2] ultralytics 설치 (캐글 기본 이미지에 없거나 구버전일 때)
!pip -q install -U ultralytics
import ultralytics, torch
ultralytics.checks()
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


In [ ]:
# [3] 업로드한 데이터 자동 탐색 + 로컬 디스크로 추출 (드라이브 직접읽기 금지 = I/O 병목 방지)
import os, glob, zipfile, shutil
INP = "/kaggle/input"
WORK = "/kaggle/working/seg_data"

zips = glob.glob(INP + "/**/*.zip", recursive=True)
if zips:
    os.makedirs(WORK, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(WORK)
    print("zip 추출:", zips[0], "→", WORK)
else:
    # 이미 폴더로 업로드된 경우: data.yaml 있는 디렉토리 사용
    yamls = glob.glob(INP + "/**/data.yaml", recursive=True)
    assert yamls, "업로드된 seg_subset(zip 또는 data.yaml 포함 폴더)를 찾지 못함 — Add Input 확인"
    WORK = os.path.dirname(yamls[0])
    print("폴더 사용:", WORK)

# data.yaml 이 하위에 있으면 그 폴더를 base 로
yml = glob.glob(WORK + "/**/data.yaml", recursive=True)
BASE = os.path.dirname(yml[0]) if yml else WORK
print("BASE =", BASE)


In [ ]:
# [4] data.yaml 을 캐글 절대경로로 재작성 + 장수 확인
import yaml, glob
cfg = {"path": BASE, "train": "images/train", "val": "images/val", "names": {0: "crack"}}
with open(os.path.join(BASE, "data.yaml"), "w") as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
print(open(os.path.join(BASE, "data.yaml")).read())
nt = len(glob.glob(BASE + "/images/train/*.*"))
nv = len(glob.glob(BASE + "/images/val/*.*"))
print(f"train {nt}장 / val {nv}장")
assert nt > 0 and nv > 0, "이미지가 안 보임 — zip 내부 구조(images/train, images/val) 확인"


In [ ]:
# [5] 학습 (yolov8s-seg). 중간 끊김 대비: 재실행 시 아래 셀에서 resume 가능
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(
    data=os.path.join(BASE, "data.yaml"),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
    patience=20, project=PROJECT, name=NAME,
    mosaic=0.7, fliplr=0.5, flipud=0.3, cos_lr=True, close_mosaic=15,
)


In [ ]:
# [5b] (선택) 세션 끊겨 재접속했을 때만 실행 — 이어서 학습
# from ultralytics import YOLO
# model = YOLO(f"{PROJECT}/{NAME}/weights/last.pt")
# model.train(resume=True)


In [ ]:
# [6] 검증 — 마스크 지표(핵심) + 박스 지표(참고)
m = model.val()
print("=== SEG (mask) — 핵심 ===")
print("mask mAP50   :", round(float(m.seg.map50), 4))
print("mask mAP50-95:", round(float(m.seg.map), 4))
print("=== BOX (참고) ===")
print("box  mAP50   :", round(float(m.box.map50), 4))
print("box  mAP50-95:", round(float(m.box.map), 4))
print("\n※ 크랙 박스 모델 0.179 와 mask mAP 를 직접 비교하지 말 것(다른 지표). 비주얼로 병행 판단.")


In [ ]:
# [7] 예측 비주얼 6장 + best.pt 다운로드용 복사
import glob, shutil
val_imgs = sorted(glob.glob(BASE + "/images/val/*.*"))[:6]
model.predict(val_imgs, save=True, project=PROJECT, name=NAME + "_pred", conf=0.25)

best = f"{PROJECT}/{NAME}/weights/best.pt"
dst = "/kaggle/working/yolov8s_seg_crack_best.pt"
shutil.copy(best, dst)
print("다운로드:", dst, "(우측 Output 에서 저장)")

from IPython.display import Image as IdImage, display
for p in sorted(glob.glob(f"{PROJECT}/{NAME}_pred/*.jpg"))[:6]:
    display(IdImage(filename=p))


## 결과 판단 가이드
- **비주얼이 1순위**: 마스크가 가는 균열선을 정확히 따라가면 → seg 채택 근거. 박스보다 위치·길이·폭 신뢰도 큼.
- **mask mAP50**: 서브셋·60ep 기준이라 절대값보단 "학습이 되긴 하는가 + 곡선 방향"을 봄. 좋으면 데스크탑에서 풀 71k 학습으로 확장.
- 다음: `best.pt` 다운로드 → 데스크탑에서 파이프라인 하이브리드(균열=seg / 면적결함=bbox) 배선 검토.
- 배선 시 `models/` 에 커밋하면 config 후보/ CD 가 자동 반영(단 seg 는 detector 경로가 별도라 배선 필요).
